In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import string
from collections import Counter
import torchvision

In [2]:
positive_sentences = [
    "this is amazing",
    "i really love this product",
    "everything works perfectly",
    "this experience was fantastic",
    "i am very happy with the result",
    "this exceeded my expectations",
    "absolutely wonderful service",
    "i enjoyed every moment",
    "highly recommended",
    "this made my day",
    "great quality and performance",
    "i am satisfied with this",
    "super easy to use",
    "this is exactly what i needed",
    "very impressive work",
    "i feel good about this",
    "this is awesome",
    "i would buy this again",
    "totally worth it",
    "i am glad i chose this",
    "excellent support team",
    "fast and reliable",
    "beautiful design",
    "this works like a charm",
    "i am impressed",
    "perfect for my needs",
    "this is outstanding",
    "i love the features",
    "great experience overall",
    "this is top quality",
    "very helpful tool",
    "this saved me a lot of time",
    "i am extremely pleased",
    "fantastic performance",
    "works better than expected",
    "i feel satisfied",
    "this is brilliant",
    "no complaints at all",
    "this is wonderful",
    "it performs very well",
    "i am happy with it",
    "superb results",
    "this is great value",
    "i really like this",
    "very user friendly",
    "this is perfect",
    "excellent quality",
    "i appreciate this",
    "this is reliable",
    "very effective solution",
    "i had a great time",
    "this is impressive",
    "love the design",
    "this is very useful",
    "i am delighted",
    "great functionality",
    "this is excellent",
    "i enjoyed using this",
    "this feels premium",
    "very good experience",
    "this is satisfying",
    "i am pleased with it",
    "great results",
    "this is fantastic",
    "works smoothly",
    "this is high quality",
    "i recommend this",
    "very positive experience",
    "this is powerful",
    "i love how it works",
    "this is outstanding",
    "very enjoyable",
    "this made things easier",
    "i am impressed with this",
    "this is solid",
    "great performance",
    "this is reliable",
    "i feel happy",
    "this is well made",
    "very nice experience",
    "this is successful",
    "i am content",
    "this is great",
    "very effective",
    "this works perfectly",
    "i am very satisfied",
    "this is amazing quality",
    "great job",
    "this is excellent work",
    "i am very happy",
    "this is a pleasure to use",
    "totally satisfied"
]

negative_sentences = [
    "i do not like this",
    "this is terrible",
    "i am very disappointed",
    "this does not work",
    "waste of time",
    "completely useless",
    "i regret buying this",
    "this is awful",
    "very bad experience",
    "i am unhappy with this",
    "this failed miserably",
    "not worth it",
    "i hate this product",
    "this is frustrating",
    "poor quality",
    "this is disappointing",
    "i would not recommend this",
    "this is broken",
    "does not meet expectations",
    "very slow and buggy",
    "this is annoying",
    "i am dissatisfied",
    "this is horrible",
    "total waste of money",
    "this does not help",
    "i feel cheated",
    "very poor performance",
    "this is unreliable",
    "i am frustrated",
    "this is unacceptable",
    "not satisfied at all",
    "this is low quality",
    "it stopped working",
    "this is a failure",
    "i do not enjoy this",
    "very bad design",
    "this is confusing",
    "i am unhappy",
    "this does not function",
    "extremely disappointing",
    "this is stressful",
    "i dislike this",
    "this is not useful",
    "i had a bad experience",
    "this is problematic",
    "it crashes often",
    "this is poorly made",
    "i am angry about this",
    "this is annoying to use",
    "this is a mess",
    "very inefficient",
    "this is useless",
    "i am not impressed",
    "this is bad",
    "this is faulty",
    "i feel disappointed",
    "this is a nightmare",
    "this is not reliable",
    "i am upset",
    "this is terrible quality",
    "this is exhausting",
    "this is misleading",
    "i feel frustrated",
    "this is not good",
    "this is poorly designed",
    "this is a problem",
    "i am not happy",
    "this is boring",
    "this is not working properly",
    "this is slow",
    "this is irritating",
    "this is unacceptable quality",
    "this is confusing and bad",
    "this is disappointing quality",
    "i would not buy this again",
    "this is broken and useless",
    "this is very bad",
    "this is horrible quality",
    "this is a waste",
    "this is poorly implemented",
    "this is frustrating to use",
    "this is annoying and slow",
    "this is not worth the price",
    "this is extremely bad",
    "this is disappointing performance",
    "this is awful experience",
    "this is not what i expected",
    "this is a disaster",
    "i am very unhappy"
]

In [3]:
def preprocess(text):
  text=text.lower()
  text=text.translate(str.maketrans('','',string.punctuation))
  return text

In [4]:
data = positive_sentences + negative_sentences
labels = [1] * len(positive_sentences) + [0] * len(negative_sentences)

In [5]:
data = [preprocess(sentence) for sentence in data]

In [6]:
#vocab creation
all_words = " ".join(data).split()
word_counts = Counter(all_words)
vocab = {word: idx+1 for idx, (word,_) in enumerate(word_counts.items())}
vocab["<PAD>"] = 0 #padding special token

In [8]:
max_len = 15 #max sentence length
def sentence_to_tensor(sentence, vocab, max_len=15):
  tokens = sentence.split()
  indices = [vocab.get(word, 0) for word in tokens]
  indices = indices[:max_len]
  indices += [0] * (max_len - len(indices))
  return torch.tensor(indices)

X = torch.stack([sentence_to_tensor(sentence, vocab, max_len) for sentence in data])
y = torch.tensor(labels)

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42 )

In [10]:
class Transformer(nn.Module):
  def __init__(self,vocab_size,embedding_dim,num_heads,num_layers,hidden_dim,num_classes):
    super(Transformer,self).__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_dim)
    self.positional_encoding = nn.Parameter(torch.randn(1,max_len,embedding_dim))
    self.transformer = nn.Transformer(d_model=embedding_dim, #embedding vector size
                                      nhead=num_heads, #head in multihead embedding
                                      num_encoder_layers=num_layers, #transformer encode layer number
                                      dim_feedforward=hidden_dim) #encoder

    self.fc = nn.Linear(embedding_dim * max_len, hidden_dim)
    self.out = nn.Linear(hidden_dim, num_classes)
    self.sigmoid = nn.Sigmoid()

  def forward(self, x):
    embedded = self.embedding(x) + self.positional_encoding
    output = self.transformer(embedded,embedded)
    output = output.view(output.size(0), -1)
    output = torch.relu(self.fc(output))
    output = self.out(output)
    output = self.sigmoid(output)
    return output

In [12]:
vocab_size = len(vocab)
embedding_dim = 32
num_heads = 4
num_layers = 4
hidden_dim = 64
num_classes = 1

model = Transformer(vocab_size,embedding_dim,num_heads,num_layers,hidden_dim,num_classes)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.0005)
epochs = 100
model.train()

for epoch in range(epochs):
  optimizer.zero_grad()
  output = model(X_train.long()).squeeze()
  loss = criterion(output, y_train.float())
  loss.backward()
  optimizer.step()

  print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item()}")

Epoch 1/100, Loss: 0.6949338912963867
Epoch 2/100, Loss: 0.6914329528808594
Epoch 3/100, Loss: 0.6995387673377991
Epoch 4/100, Loss: 0.6917685270309448
Epoch 5/100, Loss: 0.6889826059341431
Epoch 6/100, Loss: 0.7000725865364075
Epoch 7/100, Loss: 0.6848253011703491
Epoch 8/100, Loss: 0.691055178642273
Epoch 9/100, Loss: 0.6887297034263611
Epoch 10/100, Loss: 0.6875069737434387
Epoch 11/100, Loss: 0.6883079409599304
Epoch 12/100, Loss: 0.6826221942901611
Epoch 13/100, Loss: 0.6783978939056396
Epoch 14/100, Loss: 0.6736554503440857
Epoch 15/100, Loss: 0.6842787265777588
Epoch 16/100, Loss: 0.6702319383621216
Epoch 17/100, Loss: 0.6729212999343872
Epoch 18/100, Loss: 0.6734839677810669
Epoch 19/100, Loss: 0.6624572277069092
Epoch 20/100, Loss: 0.652425229549408
Epoch 21/100, Loss: 0.6415280103683472
Epoch 22/100, Loss: 0.6385101675987244
Epoch 23/100, Loss: 0.6508317589759827
Epoch 24/100, Loss: 0.6383842825889587
Epoch 25/100, Loss: 0.6373326182365417
Epoch 26/100, Loss: 0.62481987476348

In [14]:
model.eval()
with torch.no_grad():
  y_pred = model(X_test.long()).squeeze()
  y_pred = (y_pred > 0.5).float()

  y_pred_training = model(X_train.long()).squeeze()
  y_pred_training = (y_pred_training > 0.5).float()

accuracy = accuracy_score(y_test, y_pred)
train_acc = accuracy_score(y_train, y_pred_training)
print(f"Accuracy: {accuracy}")
print(f"Train accuracy: {train_acc}")

Accuracy: 0.6756756756756757
Train accuracy: 0.9027777777777778
